# Notebook 12 — Preparación de la matriz multivariante

## Finalidad

En este notebook preparamos la matriz que usaremos en los análisis multivariantes posteriores.

Partimos de las 74 variables revisadas en el Notebook 11. Analizamos posibles redundancias y la necesidad de aplicar transformaciones. Después seleccionamos las variables que guardaremos y las situaremos en una escala comparable.

Todas las decisiones de mantenimiento, transformación o exclusión quedan registradas para conservar la trazabilidad.

El análisis y los ajustes los realizamos sobre las 22 regiones NUTS 2 comparables. Andorra no forma parte de esta matriz y la mantenemos en el proyecto como unidad complementaria, sin intervenir en la selección ni en el ajuste de las transformaciones.

In [1]:
# Carga de las entradas corregidas
from pathlib import Path
import numpy as np
import pandas as pd

base_01a = Path("/kaggle/input/datasets/aemjeloy")
nb10_01a = (
    base_01a
    / "ntb-10-corregido"
    / "paquete_analisis_notebook_11"
)
nb11_01a = base_01a / "eda-11-to-12"
ruta_matriz_01a = (
    nb10_01a
    / "01_datos"
    / "dataset_analitico_comparable_22x78.parquet"
)
ruta_catalogo_01a = (
    nb10_01a
    / "02_documentacion"
    / "catalogo_tecnico_maestro_territorial_113_columnas.csv"
)
ruta_descriptivos_01a = (
    nb11_01a
    / "resultados"
    / "resumen_descriptivo_74_variables.csv"
)
ruta_iqr_01a = (
    nb11_01a
    / "resultados"
    / "resumen_valores_extremos_iqr.csv"
)
ruta_spearman_01a = (
    nb11_01a
    / "resultados"
    / "correlaciones_spearman_pares.csv"
)
rutas_01a = [
    ruta_matriz_01a,
    ruta_catalogo_01a,
    ruta_descriptivos_01a,
    ruta_iqr_01a,
    ruta_spearman_01a,
]
if not all(ruta.exists() for ruta in rutas_01a):
    raise FileNotFoundError(
        "No se encuentran todas las entradas corregidas."
    )
df_comparable = pd.read_parquet(ruta_matriz_01a)
df_catalogo = pd.read_csv(ruta_catalogo_01a)
df_descriptivos = pd.read_csv(ruta_descriptivos_01a)
df_iqr = pd.read_csv(ruta_iqr_01a)
df_spearman = pd.read_csv(ruta_spearman_01a)

electricas_01a = [
    columna
    for columna in df_comparable
    if columna.startswith("electrica_")
]
galicia_01a = df_comparable.loc[
    df_comparable["codigo_geo"].eq("ES11"),
    "electrica_densidad_lineas_km_1000km2",
].iloc[0]

if (
    df_comparable.shape != (22, 78)
    or len(df_catalogo) != 113
    or len(df_descriptivos) != 74
    or len(df_iqr) != 74
    or len(df_spearman) != 2701
    or len(electricas_01a) != 6
    or df_comparable[electricas_01a].eq(0).any().any()
    or not np.isclose(galicia_01a, 423.563476)
    or df_iqr["n_extremos"].gt(0).sum() != 44
    or df_spearman["correlacion_abs"].ge(0.90).sum() != 39
    or df_spearman["correlacion_abs"].ge(0.95).sum() != 14
):
    raise ValueError(
        "Las entradas corregidas necesitan revisión."
    )
OUTPUT_DIR = Path("/kaggle/working/salidas_notebook_12")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Matriz:", df_comparable.shape)
print("EDA: 74 variables | 2701 parejas")
print("Variables eléctricas: 6 | Ceros: 0")


Matriz: (22, 78)
EDA: 74 variables | 2701 parejas
Variables eléctricas: 6 | Ceros: 0


### Comprobación de la matriz de partida

Comprobamos que la matriz contiene 22 territorios comparables y 74 variables analíticas numéricas, sin duplicados ni valores ausentes.

También revisamos que las variables coincidan con el catálogo técnico generado en el Notebook 10.

In [2]:
# Comprobación estructural
COLUMNAS_ID = [
    "codigo_geo",
    "nombre_territorio",
    "pais",
    "ambito_modelo",
]
variables_analiticas = (
    df_comparable.columns
    .drop(COLUMNAS_ID)
    .tolist()
)
catalogo_analitico = df_catalogo[
    df_catalogo["variable"].isin(variables_analiticas)
].copy()

if (
    df_comparable.shape != (22, 78)
    or df_comparable["codigo_geo"].nunique() != 22
    or df_comparable["codigo_geo"].eq("AD").any()
    or df_comparable[variables_analiticas].isna().any().any()
    or not all(
        pd.api.types.is_numeric_dtype(df_comparable[c])
        for c in variables_analiticas
    )
    or set(catalogo_analitico["variable"])
       != set(variables_analiticas)
    or not catalogo_analitico[
        "tipo_variable"
    ].eq("Analítica").all()
    or catalogo_analitico["familia"].nunique() != 9
):
    raise ValueError("La matriz necesita revisión.")

print("Territorios: 22 | Variables: 74 | Familias: 9")


Territorios: 22 | Variables: 74 | Familias: 9


### Trazabilidad con el EDA

Comprobamos que los descriptivos, los valores extremos y las correlaciones del Notebook 11 corresponden a las mismas 74 variables de la matriz.

También verificamos que Spearman contiene las 2.701 parejas posibles, sin duplicados ni variables desconocidas.

In [3]:
# Trazabilidad con el EDA
variables_matriz = set(variables_analiticas)
parejas_01c = [
    frozenset(pareja)
    for pareja in df_spearman[
        ["variable_1", "variable_2"]
    ].itertuples(index=False, name=None)
]

if (
    set(df_descriptivos["variable"]) != variables_matriz
    or set(df_iqr["variable"]) != variables_matriz
    or len(parejas_01c) != 2701
    or any(len(pareja) != 2 for pareja in parejas_01c)
    or len(set(parejas_01c)) != 2701
    or set().union(*parejas_01c) != variables_matriz
):
    raise ValueError("La trazabilidad con el EDA necesita revisión.")

print("Variables: 74 | Parejas Spearman: 2701")


Variables: 74 | Parejas Spearman: 2701


## Selección y registro de decisiones

La reducción de variables combina tres tipos de evidencia: relaciones estructurales entre indicadores, resultados del EDA y correlaciones de Spearman. Después revisamos cada familia temática por separado.

Para cada una de las 74 variables creamos un registro con su decisión, estado de transformación y justificación. Inicialmente estos campos quedan pendientes y los actualizamos durante la revisión por familias hasta determinar qué variables mantenemos o excluimos y qué transformación corresponde en cada caso.

In [4]:
# Registro inicial de decisiones
df_decisiones = (
    catalogo_analitico
    .merge(
        df_descriptivos[
            ["variable", "asimetria", "n_ceros", "pct_ceros"]
        ],
        on="variable",
        validate="one_to_one",
    )
    .merge(
        df_iqr[
            ["variable", "n_extremos", "pct_extremos"]
        ],
        on="variable",
        validate="one_to_one",
    )
    .sort_values("orden")
    .reset_index(drop=True)
    .assign(
        decision="PENDIENTE",
        transformacion="PENDIENTE",
        motivo_decision="",
        evidencia_decision="",
    )
)
evidencias_02a = [
    "asimetria",
    "n_ceros",
    "pct_ceros",
    "n_extremos",
    "pct_extremos",
]
if (
    len(df_decisiones) != 74
    or df_decisiones["variable"].duplicated().any()
    or df_decisiones[evidencias_02a].isna().any().any()
):
    raise ValueError("El registro necesita revisión.")

print("Variables: 74 | Familias: 9 | Pendientes: 74")


Variables: 74 | Familias: 9 | Pendientes: 74


### Revisión de relaciones estructurales

Antes de utilizar las correlaciones para reducir variables, comprobamos si algunos indicadores están relacionados directamente por su forma de cálculo.

Esto permite distinguir una correlación territorial elevada de una redundancia matemática real, por ejemplo cuando una variable es una suma, una media o una transformación directa de otras.

In [5]:
# Relaciones estructurales
datos_02b = df_comparable
def sumar_02b(*columnas):
    return datos_02b[list(columnas)].sum(axis=1)
relaciones_02b = {
    "Precipitación total/media": (
        datos_02b["precipitation_total_mm_2024_2025"],
        datos_02b["precipitation_daily_mean_mm_2024_2025"] * 731,
    ),
    "Radiación total/media": (
        datos_02b["solar_radiation_total_mj_m2_2024_2025"],
        datos_02b[
            "solar_radiation_daily_mean_mj_m2_2024_2025"
        ] * 731,
    ),
    "PVGIS anual/mensual": (
        datos_02b["pvgis_produccion_anual_kwh_kwp"],
        datos_02b[
            "pvgis_produccion_media_mensual_kwh_kwp"
        ] * 12,
    ),
    "PVGIS amplitud": (
        datos_02b["pvgis_amplitud_mensual_kwh_kwp"],
        datos_02b["pvgis_produccion_max_mensual_kwh_kwp"]
        - datos_02b["pvgis_produccion_min_mensual_kwh_kwp"],
    ),
    "PVGIS coeficiente de variación": (
        datos_02b["pvgis_coef_variacion_mensual"],
        datos_02b["pvgis_desviacion_mensual_kwh_kwp"]
        / datos_02b["pvgis_produccion_media_mensual_kwh_kwp"],
    ),
    "Places total": (
        datos_02b["logistica_places_total_n"],
        sumar_02b(
            "logistica_city_n",
            "logistica_town_n",
            "logistica_village_n",
            "logistica_hamlet_n",
        ),
    ),
    "Road total": (
        datos_02b["logistica_road_total_km"],
        datos_02b["logistica_road_alta_capacidad_km"]
        + datos_02b["logistica_road_secondary_km"],
    ),
    "CORINE coberturas": (
        sumar_02b(
            "corine_superficies_artificiales_pct",
            "corine_superficies_agricolas_pct",
            "corine_zonas_forestales_naturales_pct",
            "corine_humedales_pct",
            "corine_masas_agua_pct",
        ),
        100,
    ),
    "CORINE compatibilidad": (
        sumar_02b(
            "corine_excluido_pct",
            "corine_compatibilidad_baja_pct",
            "corine_compatibilidad_media_pct",
            "corine_compatibilidad_alta_pct",
        ),
        100,
    ),
    "CORINE índice": (
        datos_02b["corine_compat_media_0_3"],
        (
            datos_02b["corine_compatibilidad_baja_pct"]
            + 2 * datos_02b["corine_compatibilidad_media_pct"]
            + 3 * datos_02b["corine_compatibilidad_alta_pct"]
        ) / 100,
    ),
    "Partición ambiental": (
        sumar_02b(
            "ambiental_area_protegida_pct",
            "ambiental_anillo_0_2500m_pct",
            "ambiental_anillo_2500_5000m_pct",
            "ambiental_anillo_5000_10000m_pct",
            "ambiental_fuera_10000m_pct",
        ),
        100,
    ),
    "Penalización ambiental": (
        datos_02b["ambiental_penalizacion_media_0_1"],
        (
            datos_02b["ambiental_area_protegida_pct"]
            + 0.75 * datos_02b["ambiental_anillo_0_2500m_pct"]
            + 0.50 * datos_02b["ambiental_anillo_2500_5000m_pct"]
            + 0.25 * datos_02b["ambiental_anillo_5000_10000m_pct"]
        ) / 100,
    ),
}
resultados_02b = []
for nombre, (observado, esperado) in relaciones_02b.items():
    observado = np.asarray(observado, dtype=float)
    error = np.abs(observado - np.asarray(esperado))
    error_pct = np.divide(
        error * 100,
        np.abs(observado),
        out=np.zeros_like(error),
        where=observado != 0,
    )
    resultados_02b.append({
        "relacion": nombre,
        "error_max_pct": error_pct.max(),
        "coincide": np.allclose(
            observado,
            esperado,
            rtol=1e-6,
            atol=1e-6,
        ),
    })
df_relaciones_estructurales = pd.DataFrame(resultados_02b)
print(
    "Relaciones: 12 | Coincidencias:",
    int(df_relaciones_estructurales["coincide"].sum()),
)
display(df_relaciones_estructurales.round(4))


Relaciones: 12 | Coincidencias: 10


,relacion,error_max_pct,coincide
0,Precipitación total/media,0.0000,True
1,Radiación total/media,0.0000,True
2,PVGIS anual/mensual,0.0014,False
3,PVGIS amplitud,0.0000,True
4,PVGIS coeficiente de variación,0.2546,False
5,Places total,0.0000,True
6,Road total,0.0000,True
7,CORINE coberturas,0.0000,True
8,CORINE compatibilidad,0.0000,True
9,CORINE índice,0.0000,True


### Interpretación de las relaciones estructurales

Comprobamos doce relaciones y diez coinciden numéricamente dentro de la tolerancia utilizada. Entre las equivalencias confirmadas están los totales climáticos respecto a sus medias diarias, la longitud viaria total respecto a sus componentes, las composiciones CORINE, la partición ambiental y los indicadores derivados de estas composiciones.

En PVGIS aparecen dos casos que no son equivalencias exactas. La producción anual es prácticamente igual a la producción media mensual multiplicada por 12, con una diferencia relativa máxima aproximada del 0,0014 %. Por su reducido tamaño, ambas variables contienen prácticamente la misma información.

El coeficiente de variación mensual presenta una diferencia máxima aproximada del 0,255 % respecto al cociente entre la desviación y la media. Por tanto, con los datos disponibles no la consideramos una equivalencia algebraica exacta y la guardamos como variable candidata para la revisión posterior de la familia PVGIS.

### Resolución de equivalencias directas

Eliminamos tres variables que duplican información:

- Conservamos las medias diarias de precipitación y radiación y excluimos sus totales.
- Conservamos la producción anual de PVGIS y excluimos la media mensual.

Las demás relaciones se revisarán dentro de cada familia antes de tomar una decisión.

In [6]:
# Decisiones por equivalencia directa
def aplicar_decisiones(decisiones):
    for variable, decision, motivo, evidencia in decisiones:
        mascara = df_decisiones["variable"].eq(variable)
        if mascara.sum() != 1:
            raise ValueError(f"Variable no localizada: {variable}")

        df_decisiones.loc[
            mascara,
            [
                "decision",
                "transformacion",
                "motivo_decision",
                "evidencia_decision",
            ],
        ] = [
            decision,
            "NINGUNA" if decision == "EXCLUIR" else "PENDIENTE",
            motivo,
            evidencia,
        ]

equivalencias_02c = [
    (
        "precipitation_total_mm_2024_2025",
        "precipitation_daily_mean_mm_2024_2025",
        "Total = media diaria × 731.",
    ),
    (
        "solar_radiation_total_mj_m2_2024_2025",
        "solar_radiation_daily_mean_mj_m2_2024_2025",
        "Total = media diaria × 731.",
    ),
    (
        "pvgis_produccion_media_mensual_kwh_kwp",
        "pvgis_produccion_anual_kwh_kwp",
        "Variables prácticamente equivalentes.",
    ),
]
for excluir, mantener, evidencia in equivalencias_02c:
    aplicar_decisiones([
        (excluir, "EXCLUIR", "Variable redundante.", evidencia),
        (mantener, "MANTENER", "Indicador más interpretable.", evidencia),
    ])
variables_02c = [
    variable
    for pareja in equivalencias_02c
    for variable in pareja[:2]
]
df_equivalencias = df_decisiones[
    df_decisiones["variable"].isin(variables_02c)
]
resultado_02c = (
    df_equivalencias["decision"].eq("MANTENER").sum(),
    df_equivalencias["decision"].eq("EXCLUIR").sum(),
    df_decisiones["decision"].eq("PENDIENTE").sum(),
)
if resultado_02c != (3, 3, 68):
    raise ValueError("Las equivalencias necesitan revisión.")
print("Mantener: 3 | Excluir: 3 | Pendientes: 68")


Mantener: 3 | Excluir: 3 | Pendientes: 68


## Selección de la familia Ambiental

Los cinco porcentajes ambientales dividen el territorio y suman el 100 %. Además, están resumidos en la penalización ambiental, por lo que mantenerlos todos introduciría información repetida.

Conservamos la penalización ambiental y la densidad de sitios protegidos. Los cinco porcentajes se excluyen de la matriz multivariante, pero se mantienen en el dataset original para su consulta.

In [7]:
# Selección de la familia Ambiental
mantener_02d = [
    "ambiental_penalizacion_media_0_1",
    "ambiental_densidad_sitios_1000km2",
]
excluir_02d = [
    "ambiental_area_protegida_pct",
    "ambiental_anillo_0_2500m_pct",
    "ambiental_anillo_2500_5000m_pct",
    "ambiental_anillo_5000_10000m_pct",
    "ambiental_fuera_10000m_pct",
]
decisiones_02d = [
    *[
        (
            variable,
            "MANTENER",
            "Indicador ambiental seleccionado.",
            "Representa restricción o densidad ambiental.",
        )
        for variable in mantener_02d
    ],
    *[
        (
            variable,
            "EXCLUIR",
            "Componente ambiental redundante.",
            "Las cinco zonas suman 100 %.",
        )
        for variable in excluir_02d
    ],
]
aplicar_decisiones(decisiones_02d)
df_ambiental = df_decisiones[
    df_decisiones["familia"].eq("Ambiental")
]
resultado_02d = df_ambiental["decision"].value_counts()
if resultado_02d.to_dict() != {"EXCLUIR": 5, "MANTENER": 2}:
    raise ValueError("La selección ambiental necesita revisión.")

print("Mantener: 2 | Excluir: 5")


Mantener: 2 | Excluir: 5


## 2.5 Selección de la familia CORINE

CORINE contiene dos composiciones cerradas al 100 %: las coberturas principales del suelo y los niveles de compatibilidad. Mantener todos sus componentes introduciría dependencia interna en la matriz.

Conservamos corine_compat_media_0_3 como resumen de la compatibilidad territorial. Las categorías de compatibilidad baja y media presentan correlaciones altas con este índice (ρ = -0,951 y ρ = 0,942), por lo que no se mantienen por separado.

De forma provisional conservamos corine_superficies_artificiales_pct. Su asociación con el índice de compatibilidad es bastante menor (ρ = 0,242), por lo que en esta fase no se considera redundante con el índice. La decisión definitiva sobre esta variable la revisaremos después junto con las correlaciones entre familias.


In [8]:
# Selección de la familia CORINE
mantener_02e = [
    "corine_compat_media_0_3",
    "corine_superficies_artificiales_pct",
]
variables_corine_02e = df_decisiones.loc[
    df_decisiones["familia"].eq("CORINE"),
    "variable",
].tolist()

excluir_02e = [
    variable
    for variable in variables_corine_02e
    if variable not in mantener_02e
]
aplicar_decisiones([
    *[
        (
            variable,
            "MANTENER",
            "Indicador CORINE seleccionado.",
            "Representa compatibilidad o artificialización.",
        )
        for variable in mantener_02e
    ],
    *[
        (
            variable,
            "EXCLUIR",
            "Componente redundante.",
            "Forma parte de una composición del 100 %.",
        )
        for variable in excluir_02e
    ],
])
rho_02e = df_comparable[
    [
        "corine_compat_media_0_3",
        "corine_compatibilidad_baja_pct",
        "corine_compatibilidad_media_pct",
        "corine_superficies_artificiales_pct",
    ]
].corr(method="spearman").iloc[0, 1:]

resultado_02e = df_decisiones.loc[
    df_decisiones["familia"].eq("CORINE"),
    "decision",
].value_counts().to_dict()
if resultado_02e != {"EXCLUIR": 8, "MANTENER": 2}:
    raise ValueError("La selección CORINE necesita revisión.")

print("Mantener: 2 | Excluir: 8")
print("Correlaciones:", rho_02e.round(3).to_dict())


Mantener: 2 | Excluir: 8
Correlaciones: {'corine_compatibilidad_baja_pct': -0.951, 'corine_compatibilidad_media_pct': 0.942, 'corine_superficies_artificiales_pct': 0.242}


### Selección de la familia PVGIS

La producción media mensual ya fue excluida por ser equivalente a la producción anual.

Conservamos la producción anual como indicador principal, ya que presenta una correlación elevada con la irradiación anual (ρ = 0,972). También mantenemos las pérdidas totales y el coeficiente de variación mensual.

Los demás indicadores mensuales se excluyen para evitar redundancias.

In [9]:
# Selección de la familia PVGIS
mantener_02f = [
    "pvgis_produccion_anual_kwh_kwp",
    "pvgis_perdidas_totales_pct",
    "pvgis_coef_variacion_mensual",
]
es_pvgis_02f = df_decisiones["familia"].eq("PVGIS")
se_mantiene_02f = (
    es_pvgis_02f
    & df_decisiones["variable"].isin(mantener_02f)
)
se_excluye_02f = es_pvgis_02f & ~se_mantiene_02f

if es_pvgis_02f.sum() != 9 or se_mantiene_02f.sum() != 3:
    raise ValueError("Las variables PVGIS necesitan revisión.")
df_decisiones.loc[
    se_mantiene_02f, "decision"
] = "MANTENER"
df_decisiones.loc[
    se_mantiene_02f, "transformacion"
] = "PENDIENTE"
df_decisiones.loc[
    se_mantiene_02f, "motivo_decision"
] = "Indicador PVGIS seleccionado."
df_decisiones.loc[
    se_mantiene_02f, "evidencia_decision"
] = "Representa producción, pérdidas o variabilidad."

df_decisiones.loc[
    se_excluye_02f, "decision"
] = "EXCLUIR"
df_decisiones.loc[
    se_excluye_02f, "transformacion"
] = "NINGUNA"
df_decisiones.loc[
    se_excluye_02f, "motivo_decision"
] = "Indicador PVGIS redundante."
df_decisiones.loc[
    se_excluye_02f, "evidencia_decision"
] = "Se conserva una representación reducida."
rho_02f = df_comparable[
    [
        "pvgis_produccion_anual_kwh_kwp",
        "pvgis_irradiacion_anual_kwh_m2",
    ]
].corr(method="spearman").iloc[0, 1]

resultado_02f = (
    se_mantiene_02f.sum(),
    se_excluye_02f.sum(),
    df_decisiones["decision"].eq("PENDIENTE").sum(),
)
if resultado_02f != (3, 6, 44):
    raise ValueError("La selección PVGIS necesita revisión.")

print("Mantener: 3 | Excluir: 6 | Pendientes: 44")
print(f"Spearman anual-irradiación: {rho_02f:.3f}")


Mantener: 3 | Excluir: 6 | Pendientes: 44
Spearman anual-irradiación: 0.972


### Selección de la familia Logística

La familia Logística contiene totales, componentes y densidades que representan información relacionada.

Para comparar territorios de distinto tamaño, conservamos la densidad de vías de alta capacidad y la densidad de núcleos de población.

Las demás variables se excluyen de la matriz reducida para evitar redundancias.

In [10]:
# Selección de la familia Logística
def seleccionar_familia(familia, mantener, evidencia):
    mascara = df_decisiones["familia"].eq(familia)
    seleccionadas = df_decisiones.loc[
        mascara, "variable"
    ].isin(mantener)

    if seleccionadas.sum() != len(mantener):
        raise ValueError(f"Revisar variables de {familia}.")
    df_decisiones.loc[mascara, "decision"] = np.where(
        seleccionadas, "MANTENER", "EXCLUIR"
    )
    df_decisiones.loc[mascara, "transformacion"] = np.where(
        seleccionadas, "PENDIENTE", "NINGUNA"
    )
    df_decisiones.loc[mascara, "motivo_decision"] = np.where(
        seleccionadas,
        "Indicador seleccionado.",
        "Indicador redundante.",
    )
    df_decisiones.loc[
        mascara, "evidencia_decision"
    ] = evidencia

    return df_decisiones.loc[
        mascara, "decision"
    ].value_counts().to_dict()

resultado_02g = seleccionar_familia(
    "Logística",
    [
        "logistica_road_alta_capacidad_km_por_1000km2",
        "logistica_places_n_por_1000km2",
    ],
    "Se priorizan indicadores normalizados por superficie.",
)
if resultado_02g != {"EXCLUIR": 11, "MANTENER": 2}:
    raise ValueError("La selección logística necesita revisión.")

print("Mantener: 2 | Excluir: 11 | Pendientes: 31")


Mantener: 2 | Excluir: 11 | Pendientes: 31


### Selección de la familia Climática

Las equivalencias entre totales y medias diarias ya se resolvieron anteriormente.

Conservamos la precipitación media diaria, la temperatura media y el punto de rocío. Excluimos la radiación diaria porque la dimensión solar ya está representada por PVGIS, y dejamos la humedad del suelo fuera de la matriz reducida.

La familia Climática queda representada por tres variables.

In [11]:
# Selección de la familia Climática
resultado_02h = seleccionar_familia(
    "Climática",
    [
        "precipitation_daily_mean_mm_2024_2025",
        "temperatura_media_2m_c",
        "punto_rocio_medio_c",
    ],
    "Se conservan precipitación, temperatura y punto de rocío.",
)
pendientes_02h = df_decisiones["decision"].eq("PENDIENTE").sum()
if resultado_02h != {"EXCLUIR": 4, "MANTENER": 3} or pendientes_02h != 28:
    raise ValueError("La selección climática necesita revisión.")

print("Mantener: 3 | Excluir: 4 | Pendientes: 28")


Mantener: 3 | Excluir: 4 | Pendientes: 28


### Selección de la familia Socioeconómica

La población total y la población en edad laboral presentan una correlación de Spearman muy elevada (ρ = 0,998). Para evitar mantener dos indicadores redundantes del tamaño poblacional, conservamos poblacion_edad_laboral y excluimos poblacion_total.

También mantenemos densidad_poblacion, tasa_desempleo y nivel_formativo_terciario, ya que representan aspectos diferentes de concentración territorial, mercado laboral y cualificación.

La familia queda reducida a cuatro variables.

In [12]:
# Selección de la familia Socioeconómica
resultado_02i = seleccionar_familia(
    "Socioeconómica",
    [
        "poblacion_edad_laboral",
        "densidad_poblacion",
        "tasa_desempleo",
        "nivel_formativo_terciario",
    ],
    "Indicadores demográficos, laborales y formativos.",
)
rho_02i = df_comparable["poblacion_total"].corr(
    df_comparable["poblacion_edad_laboral"],
    method="spearman",
)
pendientes_02i = df_decisiones["decision"].eq("PENDIENTE").sum()
if resultado_02i != {"MANTENER": 4, "EXCLUIR": 1} or pendientes_02i != 23:
    raise ValueError("La selección socioeconómica necesita revisión.")
print(
    f"Mantener: 4 | Excluir: 1 | Pendientes: 23 | "
    f"Spearman: {rho_02i:.3f}"
)


Mantener: 4 | Excluir: 1 | Pendientes: 23 | Spearman: 0.998


### Selección de la familia Hídrica

Los tres indicadores WEI+ están relacionados y representan la presión hídrica territorial.

Conservamos el porcentaje del territorio sometido a estrés grave por ser el indicador más directo. La media y el percentil 90 se excluyen de la matriz reducida, pero permanecen disponibles en el dataset original.

In [13]:
# Selección de la familia Hídrica
variables_02j = [
    "hidrica_wei_media_ponderada_pct_2019_2023",
    "hidrica_wei_p90_ponderado_pct_2019_2023",
    "hidrica_wei_area_estres_grave_pct_territorio_2019_2023",
]
seleccionada_02j = variables_02j[-1]
resultado_02j = seleccionar_familia(
    "Hídrica",
    [seleccionada_02j],
    "Se conserva el porcentaje territorial con estrés grave.",
)
rho_02j = (
    df_comparable[variables_02j]
    .corr(method="spearman")[seleccionada_02j]
    .drop(seleccionada_02j)
    .round(3)
    .to_dict()
)
pendientes_02j = df_decisiones["decision"].eq("PENDIENTE").sum()
if resultado_02j != {"EXCLUIR": 2, "MANTENER": 1} or pendientes_02j != 20:
    raise ValueError("La selección hídrica necesita revisión.")

print("Mantener: 1 | Excluir: 2 | Pendientes: 20")
print("Correlaciones:", rho_02j)


Mantener: 1 | Excluir: 2 | Pendientes: 20
Correlaciones: {'hidrica_wei_media_ponderada_pct_2019_2023': 0.892, 'hidrica_wei_p90_ponderado_pct_2019_2023': 0.9}


### Selección de la familia Eléctrica

Las seis variables eléctricas presentan cobertura completa y ningún valor cero después de corregir la fuente.

La familia contiene tres valores absolutos y sus tres versiones normalizadas por superficie. Para comparar territorios de distinto tamaño, conservamos las densidades de líneas, líneas de alta tensión y subestaciones.

Los tres valores absolutos se excluyen de la matriz reducida para evitar representar dos veces la misma infraestructura. Las posibles correlaciones entre las densidades se revisarán posteriormente.

In [14]:
# Selección de la familia Eléctrica
mantener_02k = [
    "electrica_densidad_lineas_km_1000km2",
    "electrica_densidad_alta_tension_km_1000km2",
    "electrica_densidad_subestaciones_1000km2",
]
resultado_02k = seleccionar_familia(
    "Eléctrica",
    mantener_02k,
    "Se priorizan las densidades por superficie.",
)
variables_02k = df_decisiones.loc[
    df_decisiones["familia"].eq("Eléctrica"),
    "variable",
]
ceros_02k = int(
    (df_comparable[variables_02k] == 0).sum().sum()
)
pendientes_02k = df_decisiones["decision"].eq("PENDIENTE").sum()

if (
    resultado_02k != {"MANTENER": 3, "EXCLUIR": 3}
    or ceros_02k != 0
    or pendientes_02k != 14
):
    raise ValueError("La selección eléctrica necesita revisión.")

print("Mantener: 3 | Excluir: 3 | Ceros: 0 | Pendientes: 14")


Mantener: 3 | Excluir: 3 | Ceros: 0 | Pendientes: 14


### Selección de la familia Digital

La familia Digital combina recuentos y densidades procedentes de OSM, PeeringDB y RIPE Atlas.

Conservamos la densidad de torres, la densidad de centrales de telecomunicaciones y el número de centros de datos identificados en OSM.

Las demás variables se excluyen por duplicidad, cobertura limitada o elevada presencia de ceros. La familia queda representada por tres indicadores.

In [15]:
# Selección de la familia Digital
resultado_02l = seleccionar_familia(
    "Digital",
    [
        "digital_torres_mastiles_1000km2",
        "digital_exchanges_centrales_telecom_1000km2",
        "digital_data_centers_osm_n",
    ],
    "Tres indicadores digitales complementarios.",
)
if (
    resultado_02l != {"EXCLUIR": 11, "MANTENER": 3}
    or df_decisiones["decision"].eq("PENDIENTE").any()
):
    raise ValueError("La selección digital necesita revisión.")

print("Mantener: 3 | Excluir: 11 | Pendientes: 0")


Mantener: 3 | Excluir: 11 | Pendientes: 0


### Revisión global de redundancias y selección final

Después de revisar cada familia quedan 23 variables provisionales. Entre ellas detectamos ocho parejas con `|ρ| ≥ 0,80`.

Excluimos `corine_superficies_artificiales_pct` por su relación con la densidad de población (ρ = 0,947) y la densidad viaria (ρ = 0,868).

También excluimos `electrica_densidad_lineas_km_1000km2`, relacionada con la densidad de torres digitales (ρ = 0,945) y la densidad de subestaciones (ρ = 0,823). Conservamos los indicadores eléctricos de alta tensión y subestaciones por ser más específicos.

Las demás correlaciones elevadas se mantienen porque representan dimensiones diferentes.

La selección final queda formada por 21 variables de las nueve familias analíticas.

In [16]:
# Revisión global y selección final
excluir_02m = [
    "corine_superficies_artificiales_pct",
    "electrica_densidad_lineas_km_1000km2",
]
preseleccion_02m = set(
    df_decisiones.loc[
        df_decisiones["decision"].eq("MANTENER"),
        "variable",
    ]
) | set(excluir_02m)
pares_02m = df_spearman[
    df_spearman["variable_1"].isin(preseleccion_02m)
    & df_spearman["variable_2"].isin(preseleccion_02m)
]
redundancias_02m = pares_02m[
    pares_02m["correlacion_abs"].ge(0.80)
].sort_values("correlacion_abs", ascending=False)

aplicar_decisiones([
    (
        variable,
        "EXCLUIR",
        "Redundancia global elevada.",
        "Se conserva otro indicador más específico.",
    )
    for variable in excluir_02m
])
variables_finales = df_decisiones.loc[
    df_decisiones["decision"].eq("MANTENER"),
    "variable",
].tolist()

control_02m = (
    len(preseleccion_02m),
    len(pares_02m),
    len(redundancias_02m),
    len(variables_finales),
    df_decisiones.loc[
        df_decisiones["decision"].eq("MANTENER"),
        "familia",
    ].nunique(),
)
if control_02m != (23, 253, 8, 21, 9):
    raise ValueError("La selección final necesita revisión.")

print("Preselección: 23 | Correlaciones altas: 8")
print("Selección final: 21 variables | 9 familias")
print("Estado 02M: correcto")

display(
    redundancias_02m[
        ["variable_1", "variable_2", "correlacion_spearman"]
    ].round(3)
)

Preselección: 23 | Correlaciones altas: 8
Selección final: 21 variables | 9 familias
Estado 02M: correcto


,variable_1,variable_2,correlacion_spearman
14,densidad_poblacion,corine_superficies_artificiales_pct,0.947
15,electrica_densidad_lineas_km_1000km2,digital_torres_mastiles_1000km2,0.945
51,pvgis_produccion_anual_kwh_kwp,precipitation_daily_mean_mm_2024_2025,-0.882
59,electrica_densidad_subestaciones_1000km2,digital_torres_mastiles_1000km2,0.869
60,corine_superficies_artificiales_pct,logistica_road_alta_capacidad_km_por_1000km2,0.868
88,electrica_densidad_lineas_km_1000km2,electrica_densidad_subestaciones_1000km2,0.823
96,tasa_desempleo,electrica_densidad_lineas_km_1000km2,-0.809
101,pvgis_produccion_anual_kwh_kwp,pvgis_coef_variacion_mensual,-0.803


## Preparación de la matriz multivariante

Una vez seleccionadas las 21 variables, revisamos si alguna necesita una transformación antes de la estandarización.

No buscamos obtener distribuciones normales, sino reducir asimetrías fuertes que puedan influir en los análisis posteriores.

### Diagnóstico previo

Revisamos el rango, la asimetría, los ceros y los valores negativos de cada variable.

Utilizamos `|asimetría| ≥ 1` como señal de revisión, no como una regla automática de transformación.

In [17]:
# Diagnóstico previo a las transformaciones
variables_finales = df_decisiones.loc[
    df_decisiones["decision"].eq("MANTENER"),
    "variable",
].tolist()

columnas_03a = [
    "variable", "minimo", "mediana", "maximo",
    "asimetria", "n_ceros", "pct_ceros",
]
df_diagnostico = df_descriptivos[
    df_descriptivos["variable"].isin(variables_finales)
][columnas_03a].copy()
df_diagnostico["n_negativos"] = [
    int((df_comparable[variable] < 0).sum())
    for variable in df_diagnostico["variable"]
]
df_diagnostico["permite_log1p"] = df_diagnostico["minimo"].ge(0)
df_diagnostico["revisar_transformacion"] = (
    df_diagnostico["asimetria"].abs().ge(1)
)
df_diagnostico = df_diagnostico.sort_values(
    "asimetria", key=abs, ascending=False
)
control_03a = [
    len(df_diagnostico),
    df_diagnostico["revisar_transformacion"].sum(),
    df_diagnostico["n_ceros"].gt(0).sum(),
    df_diagnostico["n_negativos"].gt(0).sum(),
]

if control_03a != [21, 12, 2, 1]:
    raise ValueError("El diagnóstico necesita revisión.")

print("Variables: 21 | Asimetría elevada: 12")
print("Con ceros: 2 | Con negativos: 1")
display(df_diagnostico.round(3))

Variables: 21 | Asimetría elevada: 12
Con ceros: 2 | Con negativos: 1


,variable,minimo,mediana,maximo,asimetria,n_ceros,pct_ceros,n_negativos,permite_log1p,revisar_transformacion
57,digital_torres_mastiles_1000km2,4.692,26.608,659.094,3.160,0,0.000,0,True,True
2,densidad_poblacion,17.700,97.400,1620.000,3.010,0,0.000,0,True,True
27,logistica_places_n_por_1000km2,14.830,88.852,1219.862,2.750,0,0.000,0,True,True
58,digital_exchanges_centrales_telecom_1000km2,0.000,0.329,3.348,2.433,2,9.091,0,True,True
26,logistica_road_alta_capacidad_km_por_1000km2,110.185,249.088,1031.779,2.298,0,0.000,0,True,True
49,electrica_densidad_subestaciones_1000km2,7.691,25.852,368.967,1.872,0,0.000,0,True,True
1,poblacion_edad_laboral,209251.000,1043684.000,5804984.000,1.574,0,0.000,0,True,True
52,digital_data_centers_osm_n,0.000,2.000,8.000,1.218,4,18.182,0,True,True
43,ambiental_penalizacion_media_0_1,0.244,0.664,0.754,-1.198,0,0.000,0,True,True
48,electrica_densidad_alta_tension_km_1000km2,45.570,107.532,283.976,1.136,0,0.000,0,True,True


### Evaluación de `log1p`

Evaluamos `log1p`, definida como `ln(1 + x)`, en las variables no negativas con asimetría positiva igual o superior a 1.

Esta transformación admite ceros y reduce la influencia de los valores muy altos. Comparamos la asimetría antes y después, pero no aplicamos la transformación automáticamente.

In [18]:
# Comparación de asimetría con LOG1P
candidatas_03b = df_diagnostico.loc[
    df_diagnostico["asimetria"].ge(1)
    & df_diagnostico["permite_log1p"],
    "variable",
].tolist()

df_log1p = pd.DataFrame({
    "variable": candidatas_03b,
    "asimetria_original": [
        df_comparable[variable].skew()
        for variable in candidatas_03b
    ],
    "asimetria_log1p": [
        np.log1p(df_comparable[variable]).skew()
        for variable in candidatas_03b
    ],
})
df_log1p["mejora_abs"] = (
    df_log1p["asimetria_log1p"].abs()
    < df_log1p["asimetria_original"].abs()
)
df_log1p = df_log1p.sort_values(
    "asimetria_original",
    ascending=False,
)

if len(df_log1p) != 11 or not df_log1p["mejora_abs"].all():
    raise ValueError("La evaluación de LOG1P necesita revisión.")

print("Variables evaluadas: 11 | Mejoran: 11")
display(df_log1p.round(3))

Variables evaluadas: 11 | Mejoran: 11


,variable,asimetria_original,asimetria_log1p,mejora_abs
0,digital_torres_mastiles_1000km2,3.160,0.762,True
1,densidad_poblacion,3.010,0.562,True
2,logistica_places_n_por_1000km2,2.750,0.288,True
3,digital_exchanges_centrales_telecom_1000km2,2.433,1.577,True
4,logistica_road_alta_capacidad_km_por_1000km2,2.298,0.573,True
5,electrica_densidad_subestaciones_1000km2,1.872,0.853,True
6,poblacion_edad_laboral,1.574,0.258,True
7,digital_data_centers_osm_n,1.218,-0.103,True
8,electrica_densidad_alta_tension_km_1000km2,1.136,0.237,True
9,hidrica_wei_area_estres_grave_pct_territorio_2...,1.095,-0.705,True


### Decisión final de transformaciones

Las once variables evaluadas reducen su asimetría con `log1p`, pero no aplicamos la transformación automáticamente.

Transformamos nueve variables de recuento o densidad, incluidas las densidades eléctricas de alta tensión y subestaciones.

La tasa de desempleo y el porcentaje territorial con estrés hídrico grave permanecen en su escala original por ser porcentajes interpretables con una asimetría moderada.

En total, aplicamos `log1p` a 9 de las 21 variables finales.

In [19]:
# 03C. Decisión final de transformaciones
variables_log1p = [
    "poblacion_edad_laboral",
    "densidad_poblacion",
    "logistica_road_alta_capacidad_km_por_1000km2",
    "logistica_places_n_por_1000km2",
    "electrica_densidad_alta_tension_km_1000km2",
    "electrica_densidad_subestaciones_1000km2",
    "digital_torres_mastiles_1000km2",
    "digital_exchanges_centrales_telecom_1000km2",
    "digital_data_centers_osm_n",
]
es_final_03c = df_decisiones["decision"].eq("MANTENER")
df_decisiones.loc[
    es_final_03c, "transformacion"
] = np.where(
    df_decisiones.loc[
        es_final_03c, "variable"
    ].isin(variables_log1p),
    "LOG1P",
    "NINGUNA",
)
df_transformaciones = df_decisiones.loc[
    es_final_03c,
    ["variable", "familia", "transformacion"],
]
resultado_03c = (
    df_transformaciones["transformacion"]
    .value_counts()
    .to_dict()
)
if resultado_03c != {"NINGUNA": 12, "LOG1P": 9}:
    raise ValueError("Las transformaciones necesitan revisión.")

print("LOG1P: 9 | Sin transformación: 12")


LOG1P: 9 | Sin transformación: 12


### Construcción de las matrices analíticas

Construimos una matriz con los valores originales y otra con las transformaciones seleccionadas.

Ambas contienen 22 territorios y 21 variables. En la segunda aplicamos `log1p` únicamente a las nueve variables indicadas.

Comprobamos que la transformación no genere valores nulos ni infinitos antes de realizar la estandarización.

In [20]:
# Construcción de las matrices analíticas
es_final_03d = (
    df_decisiones["decision"] == "MANTENER"
)
variables_finales = df_decisiones.loc[
    es_final_03d,
    "variable",
].tolist()

variables_log1p = df_decisiones.loc[
    es_final_03d
    & (df_decisiones["transformacion"] == "LOG1P"),
    "variable",
].tolist()

matriz_original = df_comparable[
    variables_finales
].copy()

matriz_transformada = matriz_original.copy()
matriz_transformada[variables_log1p] = np.log1p(
    matriz_transformada[variables_log1p]
)
control_03d = (
    matriz_original.shape,
    matriz_transformada.shape,
    len(variables_log1p),
    np.isfinite(
        matriz_transformada.to_numpy()
    ).all(),
)
if control_03d != (
    (22, 21),
    (22, 21),
    9,
    True,
):
    raise ValueError(
        "Las matrices necesitan revisión."
    )

print("Matrices: (22, 21) | LOG1P: 9 | Nulos: 0")


Matrices: (22, 21) | LOG1P: 9 | Nulos: 0


### Estandarización de la matriz

Las 21 variables utilizan escalas diferentes. Aplicamos `StandardScaler` para que todas tengan media 0 y desviación estándar 1.

El escalador se ajusta únicamente con los 22 territorios comparables; Andorra no participa.

La estandarización no convierte las variables en normales, solo permite compararlas en una misma escala.

In [21]:
# Estandarización con StandardScaler
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
matriz_estandarizada = pd.DataFrame(
    scaler.fit_transform(matriz_transformada),
    index=matriz_transformada.index,
    columns=matriz_transformada.columns,
)
df_parametros_scaler = pd.DataFrame({
    "variable": variables_finales,
    "media_ajuste": scaler.mean_,
    "desviacion_ajuste": scaler.scale_,
})
media_max_03e = matriz_estandarizada.mean().abs().max()
error_std_03e = (
    matriz_estandarizada.std(ddof=0) - 1
).abs().max()

if (
    matriz_estandarizada.shape != (22, 21)
    or not np.isfinite(matriz_estandarizada.to_numpy()).all()
    or media_max_03e > 1e-10
    or error_std_03e > 1e-10
):
    raise ValueError("La estandarización necesita revisión.")

print(
    f"Matriz: (22, 21) | "
    f"Error media: {media_max_03e:.2e} | "
    f"Error desviación: {error_std_03e:.2e}"
)
print("Estado 03E: correcto")

Matriz: (22, 21) | Error media: 2.10e-15 | Error desviación: 2.22e-16
Estado 03E: correcto


## Exportación y cierre

Guardamos los resultados del Notebook 12 para utilizarlos en los análisis posteriores.

### Exportación de los resultados

Exportamos las tres matrices, el registro de decisiones, la selección final y los parámetros de `StandardScaler`.

Al incorporar los cuatro campos de identificación, las matrices pasan de `22 × 21` a `22 × 25`.

Antes de exportar limpiamos la carpeta de salida para evitar mezclar archivos antiguos.

In [22]:
# Exportación de los resultados principales
import json
import joblib
import shutil

# Limpiar la carpeta de salida.
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

# Añadir los cuatro campos de identificación.
df_id = df_comparable[COLUMNAS_ID].reset_index(drop=True)
matrices_exportar = {
    "matriz_original_22x25": matriz_original,
    "matriz_transformada_22x25": matriz_transformada,
    "matriz_estandarizada_22x25": matriz_estandarizada,
}
for nombre, matriz in matrices_exportar.items():
    matriz_salida = pd.concat(
        [
            df_id,
            matriz.reset_index(drop=True),
        ],
        axis=1,
    )

    if matriz_salida.shape != (22, 25):
        raise ValueError(f"Dimensión incorrecta en {nombre}.")
    matriz_salida.to_csv(
        OUTPUT_DIR / f"{nombre}.csv",
        index=False,
    )
    matriz_salida.to_parquet(
        OUTPUT_DIR / f"{nombre}.parquet",
        index=False,
    )

# Documentación de la selección.
df_seleccion_final = df_decisiones.loc[
    df_decisiones["decision"] == "MANTENER",
    [
        "variable",
        "familia",
        "transformacion",
        "motivo_decision",
        "evidencia_decision",
    ],
].copy()

df_resumen_familias = (
    df_decisiones.groupby("familia")["decision"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)
df_decisiones.to_csv(
    OUTPUT_DIR / "registro_decisiones_74_variables.csv",
    index=False,
)
df_seleccion_final.to_csv(
    OUTPUT_DIR / "seleccion_final_21_variables.csv",
    index=False,
)
df_resumen_familias.to_csv(
    OUTPUT_DIR / "resumen_seleccion_por_familia.csv",
    index=False,
)
df_parametros_scaler.to_csv(
    OUTPUT_DIR / "parametros_standardscaler.csv",
    index=False,
)

# Configuración y escalador.
configuracion = {
    "territorios_ajuste": 22,
    "variables_iniciales": 74,
    "variables_finales": 21,
    "n_variables_log1p": 9,
    "variables_log1p": variables_log1p,
    "estandarizacion": "StandardScaler",
}

with open(
    OUTPUT_DIR / "configuracion_preparacion.json",
    "w",
    encoding="utf-8",
) as archivo:
    json.dump(
        configuracion,
        archivo,
        indent=4,
        ensure_ascii=False,
    )
joblib.dump(
    scaler,
    OUTPUT_DIR / "standard_scaler_21_variables.joblib",
)
n_archivos_04a = len(list(OUTPUT_DIR.iterdir()))

if len(df_seleccion_final) != 21 or n_archivos_04a != 12:
    raise ValueError("La exportación necesita revisión.")

print("Matrices: 3 | Dimensión con identificación: (22, 25)")
print("Selección final: 21 variables | Archivos: 12")


Matrices: 3 | Dimensión con identificación: (22, 25)
Selección final: 21 variables | Archivos: 12


### Validación final y creación del paquete

Comprobamos que los archivos exportados existan, tengan contenido y que las tres matrices mantengan la dimensión `22 × 25`.

Después creamos un manifiesto y agrupamos todas las salidas en un archivo ZIP para facilitar su reutilización en los siguientes notebooks.

In [23]:
# Validación final y ZIP
from zipfile import ZipFile, ZIP_DEFLATED

ruta_manifiesto = (
    OUTPUT_DIR / "manifiesto_archivos_notebook_12.csv"
)
ruta_manifiesto.unlink(missing_ok=True)
archivos_04b = sorted(
    ruta
    for ruta in OUTPUT_DIR.iterdir()
    if ruta.is_file()
)
matrices_csv_04b = [
    OUTPUT_DIR / f"matriz_{tipo}_22x25.csv"
    for tipo in [
        "original",
        "transformada",
        "estandarizada",
    ]
]
if (
    len(archivos_04b) != 12
    or any(ruta.stat().st_size == 0 for ruta in archivos_04b)
    or any(
        pd.read_csv(ruta).shape != (22, 25)
        for ruta in matrices_csv_04b
    )
):
    raise ValueError("Los archivos exportados necesitan revisión.")
df_manifiesto = pd.DataFrame({
    "archivo": [ruta.name for ruta in archivos_04b],
    "tamano_bytes": [ruta.stat().st_size for ruta in archivos_04b],
})
df_manifiesto.to_csv(ruta_manifiesto, index=False)
RUTA_ZIP = Path(
    "/kaggle/working/"
    "paquete_final_notebook_12_preparacion_multivariante.zip"
)

with ZipFile(RUTA_ZIP, "w", ZIP_DEFLATED) as archivo_zip:
    for ruta in sorted(OUTPUT_DIR.iterdir()):
        archivo_zip.write(ruta, arcname=ruta.name)

with ZipFile(RUTA_ZIP) as archivo_zip:
    if len(archivo_zip.namelist()) != 13 or archivo_zip.testzip():
        raise ValueError("El ZIP necesita revisión.")

print("Archivos principales: 12 | ZIP: 13 archivos")
print("Matrices: 3 × (22, 25)")
print("ZIP:", RUTA_ZIP)

Archivos principales: 12 | ZIP: 13 archivos
Matrices: 3 × (22, 25)
ZIP: /kaggle/working/paquete_final_notebook_12_preparacion_multivariante.zip


## Conclusiones del Notebook 12

Hemos reducido las 74 variables revisadas en el EDA a una selección final de 21 variables pertenecientes a las nueve familias analíticas.

La familia Eléctrica mantiene cobertura completa en los 22 territorios comparables. Tras la revisión de redundancias, conservamos los indicadores de densidad de líneas de alta tensión y densidad de subestaciones.

Aplicamos log1p a nueve variables y mantuvimos las otras doce en su escala original. Después estandarizamos las 21 variables usando los 22 territorios NUTS 2 comparables, sin incluir Andorra en el ajuste.

El resultado es una matriz estandarizada de 22 × 21, preparada para los análisis multivariantes posteriores. También conservamos las matrices original y transformada, el registro de decisiones y los parámetros usados en la estandarización.